# Affective Sign Language Translation
## ResNet18 + BiLSTM + Temporal Attention

## Imports

In [ ]:

import os
import gc
import cv2
import time
import random
import pickle
import warnings
import numpy as np
import pandas as pd

from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    f1_score
)

import matplotlib.pyplot as plt
import seaborn as sns

print("Imports successful.")


## Device Setup

In [ ]:

torch.set_num_threads(os.cpu_count())

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("Using CPU")


## Seed

In [ ]:

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seeds initialized.")


## Paths

In [ ]:

USE_KAGGLE = False

LOCAL_DATA_DIR = "D:/Uni/3.junior year/Sprg26/ANN's/Affective-Sign-Language-Translation-Converting-Gestures-to-Text-with-Emotional-Context/Project_Data/processed"

KAGGLE_DATA_DIR = "/kaggle/input/YOUR_DATASET_NAME/processed"

DATA_DIR = KAGGLE_DATA_DIR if USE_KAGGLE else LOCAL_DATA_DIR

TRAIN_PKL = os.path.join(DATA_DIR, "autsl_train.pkl")
VAL_PKL   = os.path.join(DATA_DIR, "autsl_val.pkl")
TEST_PKL  = os.path.join(DATA_DIR, "autsl_test.pkl")

print(TRAIN_PKL)


## Config

In [ ]:

CONFIG = {

    "batch_size": 8 if torch.cuda.is_available() else 2,

    "epochs": 10,

    "learning_rate": 1e-4,

    "weight_decay": 1e-4,

    "dropout": 0.4,

    "hidden_size": 256,

    "lstm_layers": 2,

    "gradient_clip": 1.0,

    "early_stopping_patience": 5,

    "scheduler_patience": 2,

    "subset_size": 10000,

    "freeze_cnn": True,

    "num_workers": 2 if torch.cuda.is_available() else 0,

    "top_k": 5
}

CONFIG


## Load Raw Data

In [ ]:

with open(TRAIN_PKL, "rb") as f:
    raw = pickle.load(f)

frames_all = raw["frames"]
labels_all = raw["labels"]

print("Frames shape:", frames_all.shape)
print("Labels shape:", labels_all.shape)

print("Unique classes:", len(np.unique(labels_all)))


## Class Distribution

In [ ]:

counter = Counter(labels_all)

plt.figure(figsize=(16,5))

plt.bar(counter.keys(), counter.values())

plt.title("AUTSL Class Distribution")

plt.xlabel("Class ID")
plt.ylabel("Frequency")

plt.show()


## Random Video Visualization

In [ ]:

sample_idx = random.randint(0, len(frames_all)-1)

sample_vid = frames_all[sample_idx]

fig, axes = plt.subplots(2, 8, figsize=(18,5))

for i, ax in enumerate(axes.flatten()):

    ax.imshow(sample_vid[i])

    ax.set_title(f"frame {i}")

    ax.axis("off")

plt.suptitle(
    f"Video Sample — Class {labels_all[sample_idx]}"
)

plt.tight_layout()

plt.show()


## Gloss Map

In [ ]:

def build_gloss_map(labels):

    unique_labels = sorted(list(set(labels)))

    gloss_map = {}

    for cid in unique_labels:

        gloss_map[cid] = str(cid)

    return gloss_map

GLOSS_MAP = build_gloss_map(labels_all)

print("Gloss map created.")
print(list(GLOSS_MAP.items())[:10])


## Augmentation

In [ ]:

train_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.RandomResizedCrop(
        64,
        scale=(0.8, 1.0)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([

    transforms.ToPILImage(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


## Dataset

In [ ]:

class SignLanguageDataset(Dataset):

    def __init__(
        self,
        pkl_path,
        train=False,
        subset_size=None
    ):

        with open(pkl_path, "rb") as f:

            data = pickle.load(f)

        self.frames = np.array(data["frames"])

        self.labels = np.array(data["labels"])

        if subset_size is not None:

            subset_size = min(
                subset_size,
                len(self.labels)
            )

            indices = np.random.choice(
                len(self.labels),
                subset_size,
                replace=False
            )

            self.frames = self.frames[indices]
            self.labels = self.labels[indices]

        self.train = train

        self.transform = train_transform if train else val_transform

        print(f"Loaded {len(self.labels)} samples")

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        video = self.frames[idx]

        processed_frames = []

        for frame in video:

            frame = self.transform(frame)

            processed_frames.append(frame)

        video = torch.stack(processed_frames)

        label = torch.tensor(
            self.labels[idx]
        ).long()

        return video, label


## Load Datasets

In [ ]:

train_ds = SignLanguageDataset(
    TRAIN_PKL,
    train=True,
    subset_size=CONFIG["subset_size"]
)

val_ds = SignLanguageDataset(
    VAL_PKL,
    train=False,
    subset_size=2000
)

test_ds = SignLanguageDataset(
    TEST_PKL,
    train=False,
    subset_size=2000
)


## Dataloaders

In [ ]:

counter = Counter(train_ds.labels)

weights = []

for label in train_ds.labels:

    weights.append(
        1.0 / counter[label]
    )

weights = torch.DoubleTensor(weights)

sampler = WeightedRandomSampler(

    weights,

    num_samples=len(weights),

    replacement=True
)

train_loader = DataLoader(

    train_ds,

    batch_size=CONFIG["batch_size"],

    sampler=sampler,

    num_workers=CONFIG["num_workers"]
)

val_loader = DataLoader(

    val_ds,

    batch_size=CONFIG["batch_size"],

    shuffle=False,

    num_workers=CONFIG["num_workers"]
)

test_loader = DataLoader(

    test_ds,

    batch_size=CONFIG["batch_size"],

    shuffle=False,

    num_workers=CONFIG["num_workers"]
)

print("DataLoaders ready.")


## Model

In [ ]:

class TemporalAttention(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Linear(
            hidden_size * 2,
            1
        )

    def forward(self, lstm_output):

        weights = self.attention(
            lstm_output
        )

        weights = torch.softmax(
            weights,
            dim=1
        )

        attended = torch.sum(
            weights * lstm_output,
            dim=1
        )

        return attended, weights

class ResNet18_BiLSTM_Attention(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        resnet = models.resnet18(
            weights="DEFAULT"
        )

        self.cnn = nn.Sequential(
            *list(resnet.children())[:-1]
        )

        if CONFIG["freeze_cnn"]:

            for param in self.cnn.parameters():
                param.requires_grad = False

        self.lstm = nn.LSTM(

            input_size=512,

            hidden_size=CONFIG["hidden_size"],

            num_layers=CONFIG["lstm_layers"],

            batch_first=True,

            bidirectional=True,

            dropout=CONFIG["dropout"]
        )

        self.attention = TemporalAttention(
            CONFIG["hidden_size"]
        )

        self.classifier = nn.Sequential(

            nn.BatchNorm1d(
                CONFIG["hidden_size"] * 2
            ),

            nn.Dropout(CONFIG["dropout"]),

            nn.Linear(
                CONFIG["hidden_size"] * 2,
                256
            ),

            nn.ReLU(),

            nn.Dropout(CONFIG["dropout"]),

            nn.Linear(
                256,
                num_classes
            )
        )

    def forward(self, x):

        B, T, C, H, W = x.shape

        x = x.view(B*T, C, H, W)

        features = self.cnn(x)

        features = features.squeeze(-1).squeeze(-1)

        features = features.view(B, T, 512)

        lstm_out, _ = self.lstm(features)

        attended, attention_weights = self.attention(lstm_out)

        outputs = self.classifier(attended)

        return outputs, attention_weights


## Initialize Model

In [ ]:

num_classes = int(
    np.max(train_ds.labels)
) + 1

model = ResNet18_BiLSTM_Attention(
    num_classes
)

model = model.to(DEVICE)

print(model)


## Loss + Optimizer

In [ ]:

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.AdamW(

    model.parameters(),

    lr=CONFIG["learning_rate"],

    weight_decay=CONFIG["weight_decay"]
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=CONFIG["scheduler_patience"]
)


## Top-K Accuracy

In [ ]:

def top_k_accuracy(outputs, labels, k=5):

    _, topk = outputs.topk(k, dim=1)

    correct = topk.eq(
        labels.view(-1,1)
    )

    return correct.any(dim=1).float().mean().item()


## Train Function

In [ ]:

def train_one_epoch(model, loader):

    model.train()

    running_loss = 0

    all_preds = []
    all_labels = []

    for videos, labels in loader:

        videos = videos.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs, _ = model(videos)

        loss = criterion(outputs, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CONFIG["gradient_clip"]
        )

        optimizer.step()

        running_loss += loss.item()

        preds = outputs.argmax(1)

        all_preds.extend(preds.cpu().numpy())

        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    return running_loss / len(loader), acc


## Validation

In [ ]:

@torch.no_grad()

def validate(model, loader):

    model.eval()

    running_loss = 0

    all_preds = []
    all_labels = []

    top5_scores = []

    for videos, labels in loader:

        videos = videos.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs, _ = model(videos)

        loss = criterion(outputs, labels)

        running_loss += loss.item()

        preds = outputs.argmax(1)

        all_preds.extend(preds.cpu().numpy())

        all_labels.extend(labels.cpu().numpy())

        top5_scores.append(
            top_k_accuracy(
                outputs,
                labels,
                k=5
            )
        )

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    return (
        running_loss / len(loader),
        acc,
        np.mean(top5_scores)
    )


## Training Loop

In [ ]:

best_val = 0

train_losses = []
val_losses = []

train_accs = []
val_accs = []

for epoch in range(CONFIG["epochs"]):

    print("\n" + "="*50)

    print(
        f"Epoch {epoch+1}/{CONFIG['epochs']}"
    )

    print("="*50)

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    val_loss, val_acc, val_top5 = validate(
        model,
        val_loader
    )

    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")
    print(f"Val Top-5: {val_top5:.4f}")

    if val_acc > best_val:

        best_val = val_acc

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )

        print("Best model saved.")


## Loss Curves

In [ ]:

plt.figure(figsize=(10,5))

plt.plot(train_losses,
         label="Train")

plt.plot(val_losses,
         label="Validation")

plt.title("Loss Curves")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()

plt.show()


## Accuracy Curves

In [ ]:

plt.figure(figsize=(10,5))

plt.plot(train_accs,
         label="Train")

plt.plot(val_accs,
         label="Validation")

plt.title("Accuracy Curves")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend()

plt.show()


## Frame Attention Visualization

In [ ]:

@torch.no_grad()

def show_attention(model, dataset, idx):

    model.eval()

    video, label = dataset[idx]

    outputs, attention = model(
        video.unsqueeze(0).to(DEVICE)
    )

    attention = attention.squeeze().cpu().numpy()

    fig, axes = plt.subplots(
        2,
        8,
        figsize=(18,5)
    )

    for i, ax in enumerate(axes.flatten()):

        frame = video[i].permute(1,2,0).numpy()

        frame = (
            frame - frame.min()
        ) / (
            frame.max() - frame.min()
        )

        ax.imshow(frame)

        ax.set_title(
            f"A:{attention[i]:.2f}"
        )

        ax.axis("off")

    plt.suptitle(
        f"Temporal Attention — "
        f"Class {label.item()}"
    )

    plt.show()

sample_idx = random.randint(
    0,
    len(test_ds)-1
)

show_attention(
    model,
    test_ds,
    sample_idx
)


## Confusion Matrix

In [ ]:

@torch.no_grad()

def get_predictions(model, loader):

    model.eval()

    preds = []
    labels = []

    for videos, lbls in loader:

        videos = videos.to(DEVICE)

        outputs, _ = model(videos)

        pred = outputs.argmax(1)

        preds.extend(pred.cpu().numpy())

        labels.extend(lbls.numpy())

    return preds, labels

preds, labels = get_predictions(
    model,
    test_loader
)

cm = confusion_matrix(labels, preds)

plt.figure(figsize=(12,10))

sns.heatmap(cm)

plt.title("Confusion Matrix")

plt.show()


## Top-5 Predictions

In [ ]:

@torch.no_grad()

def predict(model, dataset, idx, top_k=5):

    model.eval()

    video, true_label = dataset[idx]

    outputs, attention = model(
        video.unsqueeze(0).to(DEVICE)
    )

    probs = torch.softmax(
        outputs,
        dim=1
    )[0].cpu().numpy()

    topk = probs.argsort()[-top_k:][::-1]

    true_word = GLOSS_MAP.get(
        int(true_label),
        str(int(true_label))
    )

    print("="*52)

    print("  SIGN LANGUAGE PREDICTION")

    print("="*52)

    print(
        f"  True sign : "
        f"{true_word} "
        f"(class {true_label})"
    )

    print(f"\n  Top-{top_k}:")

    for rank, i in enumerate(topk):

        tick = "✓" if i == true_label else " "

        bar = "█" * int(probs[i] * 40)

        print(

            f"    {tick} "

            f"#{rank+1}  "

            f"{GLOSS_MAP.get(i, str(i)):20s} "

            f"{probs[i]*100:5.1f}%  "

            f"{bar}"
        )

    pred_word = GLOSS_MAP.get(
        int(topk[0]),
        str(int(topk[0]))
    )

    correct = topk[0] == true_label

    print(

        f'\n  OUTPUT: "{pred_word}"  '

        f'[{"CORRECT ✓" if correct else "WRONG ✗"}]'
    )

    print("="*52)


## Random Test Predictions

In [ ]:

for idx in np.random.choice(
    len(test_ds),
    3,
    replace=False
):

    predict(
        model,
        test_ds,
        int(idx)
    )

    print()
